Download all molecules that have been tested against the target of interest - epidermal growth factor receptor (EGFR) kinase.

In [34]:
import math
from pathlib import Path
from zipfile import ZipFile
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
from rdkit.Chem import PandasTools
from chembl_webresource_client.new_client import new_client
import tqdm as notebook_tqdm

In [35]:
last_dir = Path(_dh[-1])
print(f"Current working directory: {last_dir}")

Current working directory: /Users/duncanfreeman/dev/bioinfo/notebooks/volkmer


In [36]:
targets_api = new_client.target
compounds_api = new_client.molecule
bioactivities_api = new_client.activity

In [37]:
uniprot_id = "P00533"
# Get target information from ChEMBL but restrict it to specified values only
targets = targets_api.get(target_components__accession=uniprot_id).only(
    "target_chembl_id", "organism", "pref_name", "target_type"
)
print(f'The type of the targets is "{type(targets)}"')
targets = pd.DataFrame.from_records(targets)
targets

The type of the targets is "<class 'chembl_webresource_client.query_set.QuerySet'>"


,organism,pref_name,target_chembl_id,target_type
0,Homo sapiens,Epidermal growth factor receptor,CHEMBL203,SINGLE PROTEIN
1,Homo sapiens,Epidermal growth factor receptor,CHEMBL203,SINGLE PROTEIN
2,Homo sapiens,Epidermal growth factor receptor and ErbB2 (HE...,CHEMBL2111431,PROTEIN FAMILY
3,Homo sapiens,Epidermal growth factor receptor,CHEMBL2363049,PROTEIN FAMILY
4,Homo sapiens,MER intracellular domain/EGFR extracellular do...,CHEMBL3137284,CHIMERIC PROTEIN
5,Homo sapiens,Protein cereblon/Epidermal growth factor receptor,CHEMBL4523680,PROTEIN-PROTEIN INTERACTION
6,Homo sapiens,EGFR/PPP1CA,CHEMBL4523747,PROTEIN-PROTEIN INTERACTION
7,Homo sapiens,von Hippel-Lindau disease tumor suppressor/Epi...,CHEMBL4523998,PROTEIN-PROTEIN INTERACTION
8,Homo sapiens,Baculoviral IAP repeat-containing protein 2/Ep...,CHEMBL4802031,PROTEIN-PROTEIN INTERACTION
9,Homo sapiens,CCN2-EGFR,CHEMBL5465557,PROTEIN-PROTEIN INTERACTION


In [38]:
target = targets.iloc[0]
target

organism                                Homo sapiens
pref_name           Epidermal growth factor receptor
target_chembl_id                           CHEMBL203
target_type                           SINGLE PROTEIN
Name: 0, dtype: str

In [ ]:
# save selected ChEMBL ID to variable for later use
chembl_id = target.target_chembl_id
print(f"The target ChEMBL ID is {chembl_id}")
# NBVAL_CHECK_OUTPUT

The target ChEMBL ID is CHEMBL203


### Get Bioactivity Data
To query bioactivity data for the target of interest:

Fetch bioactivity data for the target from ChEMBL¶
In this step, we fetch the bioactivity data and filter it to only consider

- human proteins
- bioactivity type IC50
- exact measurements (relation '=')
- binding data (assay type 'B')

In [41]:
bioactivities = bioactivities_api.filter(
    target_chembl_id=chembl_id, type="IC50", relation="=", assay_type="B"
).only(
    "activity_id",
    "assay_chembl_id",
    "assay_description",
    "assay_type",
    "molecule_chembl_id",
    "type",
    "standard_units",
    "relation",
    "standard_value",
    "target_chembl_id",
    "target_organism",
)

print(f"Length and type of bioactivities object: {len(bioactivities)}, {type(bioactivities)}")

Length and type of bioactivities object: 17686, <class 'chembl_webresource_client.query_set.QuerySet'>


In [42]:
print(f"Length and type of first element: {len(bioactivities[0])}, {type(bioactivities[0])}")
bioactivities[0]

Length and type of first element: 13, <class 'dict'>


{'activity_id': 32260,
 'assay_chembl_id': 'CHEMBL674637',
 'assay_description': 'Inhibitory activity towards tyrosine phosphorylation for the epidermal growth factor-receptor kinase',
 'assay_type': 'B',
 'molecule_chembl_id': 'CHEMBL68920',
 'relation': '=',
 'standard_units': 'nM',
 'standard_value': '41.0',
 'target_chembl_id': 'CHEMBL203',
 'target_organism': 'Homo sapiens',
 'type': 'IC50',
 'units': 'uM',
 'value': '0.041'}

### Download bioactivity data from ChEMBL

In [81]:
bioactivities_df_2 = pd.DataFrame.from_dict(bioactivities)
print(f"DataFrame shape: {bioactivities_df_2.shape}")
bioactivities_df_2.head(2)

DataFrame shape: (17686, 13)


,activity_id,assay_chembl_id,assay_description,assay_type,molecule_chembl_id,relation,standard_units,standard_value,target_chembl_id,target_organism,type,units,value
0,32260,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,B,CHEMBL68920,=,nM,41.0,CHEMBL203,Homo sapiens,IC50,uM,0.041
1,32267,CHEMBL674637,Inhibitory activity towards tyrosine phosphory...,B,CHEMBL69960,=,nM,170.0,CHEMBL203,Homo sapiens,IC50,uM,0.17


In [82]:
bioactivities_df_2.to_csv("/Users/duncanfreeman/dev/bioinfo/data/bioactivities_df_2.csv", index=False)

In [83]:
bioactivities_df_2.reset_index(drop=True, inplace=True)
bioactivities_df_2.rename(
    columns={"standard_value": "IC50", "standard_units": "units"}, inplace=True
)
print(f"Shape: {bioactivities_df_2.shape}")

Shape: (17686, 13)


In [84]:
# Fetch compound information for all unique molecule_chembl_id values in the bioactivities_query_df
compounds_provider = compounds_api.filter(
    molecule_chembl_id__in=list(bioactivities_df_2["molecule_chembl_id"])
).only("molecule_chembl_id", "molecule_structures")

In [85]:
compounds = list(compounds_provider)
#compounds = list(tqdm(compounds_provider, desc="Fetching compounds", total=len(bioactivities_query_df["molecule_chembl_id"].unique())))

In [86]:
compounds_df = pd.DataFrame.from_records(
    compounds
)
print(f"DataFrame shape: {compounds_df.shape}")

DataFrame shape: (10280, 2)


In [87]:
compounds_df.head(2)

,molecule_chembl_id,molecule_structures
0,CHEMBL6246,{'canonical_smiles': 'O=c1oc2c(O)c(O)cc3c(=O)o...
1,CHEMBL10,{'canonical_smiles': 'C[S+]([O-])c1ccc(-c2nc(-...


In [ ]:
# Drop rows with missing molecule structure entry
compounds_df.dropna(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")

DataFrame shape: (10263, 2)


In [89]:
# Delete duplicate molecules
compounds_df.drop_duplicates("molecule_chembl_id", keep="first", inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")


DataFrame shape: (10263, 2)


In [90]:
# Get molecules with canonical smiles
compounds_df.iloc[0].molecule_structures.keys()

dict_keys(['canonical_smiles', 'molfile', 'standard_inchi', 'standard_inchi_key'])

In [91]:
compounds_df2 = compounds_df.assign(
    smiles=compounds_df["molecule_structures"].map(
        lambda x: x.get("canonical_smiles") if isinstance(x, dict) else None
    )
).drop(columns=["molecule_structures"])

print(f"DataFrame shape: {compounds_df2.shape}")

DataFrame shape: (10263, 2)


In [92]:
compounds_df2.dropna(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {compounds_df2.shape}")

DataFrame shape: (10263, 2)


In [93]:
print(f"Bioactivities filtered: {bioactivities_df_2.shape[0]}")
bioactivities_df_2.columns

Bioactivities filtered: 17686


Index(['activity_id', 'assay_chembl_id', 'assay_description', 'assay_type',
       'molecule_chembl_id', 'relation', 'units', 'IC50', 'target_chembl_id',
       'target_organism', 'type', 'units', 'value'],
      dtype='str')

In [94]:
print(f"Compounds filtered: {compounds_df2.shape[0]}")
compounds_df2.columns

Compounds filtered: 10263


Index(['molecule_chembl_id', 'smiles'], dtype='str')

In [106]:
# Merge DataFrames
output_df2 = pd.merge(
    bioactivities_df_2[["molecule_chembl_id", "IC50", "units"]],
    compounds_df2,
    on="molecule_chembl_id",
)

# Reset row indices
output_df2.reset_index(drop=True, inplace=True)

print(f"Dataset with {output_df2.shape[0]} entries.")

Dataset with 17669 entries.


In [107]:
output_df2.head(2)

,molecule_chembl_id,IC50,units,units,smiles
0,CHEMBL68920,41.0,nM,uM,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...
1,CHEMBL69960,170.0,nM,uM,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...


In [108]:
output_df2 = output_df2.astype({"IC50": "float64"})

In [109]:
def convert_ic50_to_pic50(IC50_value):
    pIC50_value = 9 - math.log10(IC50_value)
    return pIC50_value
# Apply conversion to each row of the compounds DataFrame
output_df2["pIC50"] = output_df2.apply(lambda x: convert_ic50_to_pic50(x.IC50), axis=1)

In [110]:
output_df2.head(2)

,molecule_chembl_id,IC50,units,units,smiles,pIC50
0,CHEMBL68920,41.0,nM,uM,Cc1cc(C)c(/C=C2\C(=O)Nc3ncnc(Nc4ccc(F)c(Cl)c4)...,7.387216
1,CHEMBL69960,170.0,nM,uM,Cc1cc(C(=O)N2CCOCC2)[nH]c1/C=C1\C(=O)Nc2ncnc(N...,6.769551


In [111]:
# Add molecule column
PandasTools.AddMoleculeColumnToFrame(output_df2, smilesCol="smiles")

In [112]:
# Sort molecules by pIC50
output_df2.sort_values(by="pIC50", ascending=False, inplace=True)

# Reset index
output_df2.reset_index(drop=True, inplace=True)

In [113]:
output_df2.drop("smiles", axis=1).head(3)

,molecule_chembl_id,IC50,units,units,pIC50,ROMol
0,CHEMBL3353410,0.002,nM,nM,11.698970,<rdkit.Chem.rdchem.Mol object at 0x11eb5ec70>
1,CHEMBL63786,0.003,nM,nM,11.522879,<rdkit.Chem.rdchem.Mol object at 0x11e93f060>
2,CHEMBL103552,0.004,ug.mL-1,10'-3 ug/ml,11.397940,<rdkit.Chem.rdchem.Mol object at 0x11e921930>


In [115]:
output_df2.columns

Index(['molecule_chembl_id', 'IC50', 'units', 'units', 'smiles', 'pIC50',
       'ROMol'],
      dtype='str')

In [116]:
import numpy as np

unit_idx = np.flatnonzero(output_df2.columns == "units")

if len(unit_idx) != 2:
    raise ValueError(f"Expected exactly 2 'units' columns, found {len(unit_idx)}")

# Keep only rows where both units columns match
mask = output_df2.iloc[:, unit_idx[0]].eq(output_df2.iloc[:, unit_idx[1]])

# Drop the second units column by position
keep_cols = [i for i in range(output_df2.shape[1]) if i != unit_idx[1]]
output_df2 = output_df2.loc[mask].iloc[:, keep_cols].copy()

In [118]:
output_df2.columns

Index(['molecule_chembl_id', 'IC50', 'units', 'smiles', 'pIC50', 'ROMol'], dtype='str')

In [119]:
# Sort molecules by pIC50
output_df2.sort_values(by="pIC50", ascending=False, inplace=True)

# Reset index
output_df2.reset_index(drop=True, inplace=True)

In [120]:
output_df2.drop("smiles", axis=1).head(3)

,molecule_chembl_id,IC50,units,pIC50,ROMol
0,CHEMBL3353410,0.002,nM,11.698970,<rdkit.Chem.rdchem.Mol object at 0x11eb5ec70>
1,CHEMBL63786,0.003,nM,11.522879,<rdkit.Chem.rdchem.Mol object at 0x11e93f060>
2,CHEMBL35820,0.006,nM,11.221849,<rdkit.Chem.rdchem.Mol object at 0x11e95b840>


In [124]:
from rdkit import Chem

output_df2["smiles"] = output_df2["ROMol"].map(
    lambda mol: Chem.MolToSmiles(mol, canonical=True)
)

In [125]:
output_df2.head(2)

,molecule_chembl_id,IC50,units,smiles,pIC50,ROMol
0,CHEMBL3353410,0.002,nM,C=CC(=O)Nc1cc(Nc2nccc(-c3cn(C)c4ccccc34)n2)c(O...,11.698970,<rdkit.Chem.rdchem.Mol object at 0x11eb5ec70>
1,CHEMBL63786,0.003,nM,Brc1cccc(Nc2ncnc3cc4ccccc4cc23)c1,11.522879,<rdkit.Chem.rdchem.Mol object at 0x11e93f060>


In [126]:
output_df2.to_csv("/Users/duncanfreeman/dev/bioinfo/data/EGFR_compounds.csv", index=False)